In [17]:
import os
import shutil

# Danh sách đúng 10 giống chó cậu muốn giữ lại
keep_breeds = [
    'afghan_hound', 'airedale', 'basenji', 'beagle', 
    'bernese_mountain_dog', 'cairn', 'chow', 
    'border_terrier', 'blenheim_spaniel', 'bedlington_terrier'
]

dataset_dirs = ['../dataset/train', '../dataset/val', '../dataset/test']

for base_dir in dataset_dirs:
    if os.path.exists(base_dir):
        for breed in os.listdir(base_dir):
            breed_path = os.path.join(base_dir, breed)
            # Xóa các giống chó không nằm trong danh sách giữ lại
            if os.path.isdir(breed_path) and breed not in keep_breeds:
                shutil.rmtree(breed_path)

print("🎉 Đã lọc xong! Giờ dữ liệu chỉ còn đúng 10 giống chó.")

🎉 Đã lọc xong! Giờ dữ liệu chỉ còn đúng 10 giống chó.


In [19]:
import tensorflow as tf
import os

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATA_DIR = "../dataset"

# 1. Load datasets gốc từ thư mục
train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, "train"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)
val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, "val"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)
test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, "test"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# 2. Lấy danh sách 10 giống chó ngay tại đây (lúc này dataset còn giữ class_names)
class_names = test_ds_raw.class_names

# 3. Tiến hành tối ưu hóa hiệu suất đọc ghi dữ liệu (Cache & Prefetch)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds_raw.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds_raw.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds_raw.cache().prefetch(buffer_size=AUTOTUNE)

print(f"\n🎉 Đã load xong dữ liệu thành công!")
print(f"Các giống chó gồm: {class_names}")

Found 822 files belonging to 10 classes.
Found 99 files belonging to 10 classes.
Found 112 files belonging to 10 classes.

🎉 Đã load xong dữ liệu thành công!
Các giống chó gồm: ['afghan_hound', 'airedale', 'basenji', 'beagle', 'bedlington_terrier', 'bernese_mountain_dog', 'blenheim_spaniel', 'border_terrier', 'cairn', 'chow']


In [20]:
# 1. Khởi tạo block Data Augmentation chống học vẹt (overfitting)
data_augmentation = tf.keras.Sequential([
  tf.keras.layers.RandomFlip("horizontal"),
  tf.keras.layers.RandomRotation(0.1),
  tf.keras.layers.RandomZoom(0.1),
])

num_classes = len(class_names)

# 2. Định nghĩa cấu trúc Model CNN
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224, 224, 3)),
    data_augmentation,
    tf.keras.layers.Rescaling(1./255),

    tf.keras.layers.Conv2D(32, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5), 

    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 3. Tiến hành Huấn luyện (15 Epochs)
print("Bắt đầu quá trình huấn luyện...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)

# 4. Đánh giá mô hình sau khi train trên tập Test sạch
print("\n--- ĐÁNH GIÁ MÔ HÌNH TRÊN TẬP TEST ---")
test_loss, test_acc = model.evaluate(test_ds)
print(f"Độ chính xác trên tập Test: {test_acc * 100:.2f}%")

Bắt đầu quá trình huấn luyện...
Epoch 1/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 10s 345ms/step - accuracy: 0.1119 - loss: 2.3615 - val_accuracy: 0.0909 - val_loss: 2.2947
Epoch 2/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 8s 315ms/step - accuracy: 0.1375 - loss: 2.2810 - val_accuracy: 0.2020 - val_loss: 2.2572
Epoch 3/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 8s 314ms/step - accuracy: 0.1934 - loss: 2.1962 - val_accuracy: 0.2020 - val_loss: 2.1598
Epoch 4/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 8s 317ms/step - accuracy: 0.2397 - loss: 2.1266 - val_accuracy: 0.1919 - val_loss: 2.0989
Epoch 5/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 8s 310ms/step - accuracy: 0.2652 - loss: 2.0537 - val_accuracy: 0.2424 - val_loss: 2.1118
Epoch 6/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 8s 306ms/step - accuracy: 0.2640 - loss: 2.0352 - val_accuracy: 0.2727 - val_loss: 2.0609
Epoch 7/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 8s 323ms/step - accuracy: 0.3139 - loss: 1.9656 - val_accuracy: 0.2424 - val_loss: 2.0885
Epoch 8/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 8s 310ms/step - accuracy: 0.2981 

In [21]:
# --- THỬ NGHIỆM 2: MẠNG CNN SÂU HƠN ---
model_deep = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224, 224, 3)),
    data_augmentation,
    tf.keras.layers.Rescaling(1./255),

    tf.keras.layers.Conv2D(32, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    # Thêm 1 lớp Conv2D 256 để tăng khả năng học
    tf.keras.layers.Conv2D(256, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.5), 
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model_deep.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_deep = model_deep.fit(train_ds, validation_data=val_ds, epochs=15)
test_loss_deep, test_acc_deep = model_deep.evaluate(test_ds)
print(f"Test Accuracy Thử nghiệm 2: {test_acc_deep * 100:.2f}%")

Epoch 1/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 13s 405ms/step - accuracy: 0.0985 - loss: 2.3484 - val_accuracy: 0.1111 - val_loss: 2.2968
Epoch 2/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 9s 362ms/step - accuracy: 0.1107 - loss: 2.2991 - val_accuracy: 0.1717 - val_loss: 2.2877
Epoch 3/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 9s 365ms/step - accuracy: 0.1667 - loss: 2.2554 - val_accuracy: 0.1616 - val_loss: 2.2220
Epoch 4/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 9s 354ms/step - accuracy: 0.1861 - loss: 2.2145 - val_accuracy: 0.1111 - val_loss: 2.2637
Epoch 5/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 9s 351ms/step - accuracy: 0.2117 - loss: 2.1733 - val_accuracy: 0.1919 - val_loss: 2.1898
Epoch 6/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 9s 345ms/step - accuracy: 0.2044 - loss: 2.1497 - val_accuracy: 0.1313 - val_loss: 2.2539
Epoch 7/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 8s 327ms/step - accuracy: 0.2178 - loss: 2.1299 - val_accuracy: 0.1818 - val_loss: 2.2190
Epoch 8/15
26/26 ━━━━━━━━━━━━━━━━━━━━ 8s 326ms/step - accuracy: 0.2141 - loss: 2.1133 - val_accuracy: 0

In [22]:
# --- THỬ NGHIỆM 3: TRANSFER LEARNING (MOBILENETV2) ---
base_model = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False # Đóng băng để lấy kiến thức từ Google

model_tl = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224, 224, 3)),
    tf.keras.layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model_tl.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_tl = model_tl.fit(train_ds, validation_data=val_ds, epochs=10)
test_loss_tl, test_acc_tl = model_tl.evaluate(test_ds)
print(f"Test Accuracy Thử nghiệm 3: {test_acc_tl * 100:.2f}%")

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step

Epoch 1/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 12s 351ms/step - accuracy: 0.6521 - loss: 1.2330 - val_accuracy: 0.9495 - val_loss: 0.3096
Epoch 2/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 8s 293ms/step - accuracy: 0.9769 - loss: 0.1777 - val_accuracy: 0.9798 - val_loss: 0.1187
Epoch 3/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 8s 309ms/step - accuracy: 0.9927 - loss: 0.0799 - val_accuracy: 0.9798 - val_loss: 0.0888
Epoch 4/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 8s 312ms/step - accuracy: 1.0000 - loss: 0.0505 - val_accuracy: 0.9798 - val_loss: 0.0758
Epoch 5/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 9s 335ms/step - accuracy: 1.0000 - loss: 0.0401 - val_accuracy: 0.9798 - val_loss: 0.0659
Epoch 6/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 8s 313ms/step - accuracy: 1.0000 - loss: 0.0289 - val_accuracy: 0.9798 - val_loss: 0.0616
Epoch 7/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 8s 309ms/step - accuracy: 1.0000 - loss: 0.0231 - val_accuracy: 0.9798 - val_loss: 0.0586
Epoch 8/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 8s 327ms/step -